# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

add jrcai_corekit to path

In [3]:
import sys
sys.path.append('/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/star-instructions-tuning/jrcai_corekit/llms_corekit')

check everything is working

In [4]:
from llm.text_generator import TextGenerator

🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/message_generator.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# Constants

In [5]:
TAWJEEH_DATASET_NAME = 'ArEntail'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/ArEntail_experimental'
MODEL_PATH = "/raid_storage/shared_models/Qwen3-8B-Base"
MODEL_NAME = "Qwen3-8B"
TASK_NAME='NLI'

In [6]:
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [7]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14901,
  'tags': [],
  'name': 'A Simple Test Prompt',
  'task': {'name': 'dialect identification'},
  'status': 'DRAFT',
  'template': 'Please predict the most suitable dialect for the following text: {{arabic}}\xa0\r\n|||{{answer_choices[label]}}',
  'created_by': 'irfan',
  'dataset_name': 'arbml/AraBench_dev',
  'dataset_subset': 'default',
  'answer_choices': ['Tunisian',
   'MSA',
   'Morrocan',
   'Qatari',
   'Egyptian',
   'Lebanese'],
  'text_direction': 'ltr'},
 {'id': 14898,
  'tags': ['', 'Zero-shot COT'],
  'name': 'Prompt with zero-shot chain of thoughts',
  'task': {'name': 'claim verification'},
  'status': 'APPROVED',
  'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
  'created_by': 'ahmed',


In [8]:
len(prompts)

365

In [9]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

352

## Finetuning

### Get the dataset prompts

In [10]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

8

In [11]:
SELECTED_PROMPTS_IDS = [
    14581,
    14816,
    14818,
    14819,
    14820,
]

In [12]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [13]:
import datasets

In [14]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 1000
    })
})

### Merge the prompts

In [15]:
from jinja2 import Environment, StrictUndefined

In [16]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}\n{suffix.strip()}' # output is always the last line!

In [17]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        sample['answer_choices'] = prompt_template['answer_choices']
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e

see how the template is applied on different examples

### Perform prompt-merge on one example prompt, for experimentation

In [18]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][3]))

Welcome to the "Logic Match Game!"

Task: In this game, your challenge is to decide if the statement, "زعيم الحوثيين يعلن عن خمس جبهات لمواجهة عاصفة الحزم ," logically follows from the premise, "اليمن.. زعيم الحوثيين يدعو لرفد جبهات القتال بالمال والرجال والإرياني يؤكد فشل مفاوضات أممية مع الجماعة بشأن صافر." Think carefully: does the premise support the hypothesis, or not?

Rules: 
- If the hypothesis follows logically, answer with "entails".
- If it does not follow, answer with "not entail".

Enter your answer (entails or not entail) to complete the challenge!
not entail


In [19]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

1000.0

In [20]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/5000 [00:00<?, ?it/s]

rending Welcome to the "Logic Match Game!"

Task: In this game, your challenge is to decide if the statement, "{{ hypothesis }}," logically follows from the premise, "{{ premise }}." Think carefully: does the premise support the hypothesis, or not?

Rules: 
- If the hypothesis follows logically, answer with "entails".
- If it does not follow, answer with "not entail".

Enter your answer (entails or not entail) to complete the challenge!
|||
{{ answer_choices[label] }} sample index: 0


rending Task: Determine if the statement, "{{ hypothesis }}," logically follows from the premise, "{{ premise }}." Carefully assess whether the premise provides enough support for the hypothesis. 

Answer with "entails" if the hypothesis logically follows, or "not entail" if it does not. Provide only your answer.
|||
{{ answer_choices[label] }} sample index: 1000


rending Task: Determine if the hypothesis follows logically from the premise. Follow these steps to reach a conclusion.

Steps:
1. Understand the Premise: Carefully read and comprehend the premise to understand its main idea.
2. Examine the Hypothesis: Read the hypothesis and consider what it implies.
3. Compare for Logical Connection: Determine if the information in the premise directly supports or implies the hypothesis.

Premise: {{ premise }}
Hypothesis: {{ hypothesis }}

Answer: Based on the previous steps, respond with only one of these options (entails, not entail). Do not include any additional explanation.
|||
{{ answer_choices[label] }} sample index: 2000


rending Based on the premise: {{ premise }}, does the following hypothesis logically follow?

Hypothesis: {{ hypothesis }}

Answer with only "entails" or "not entails". No need for extra explanation
|||
{{ answer_choices[label] }} sample index: 3000


rending Given the {{premise}}, is it true that {{hypothesis}}? {{answer_choices | join(' or ')}}
|||
{{answer_choices[label]}} sample index: 4000


5000

## Finetune the LLM

In [21]:
GLOBAL_SEED = 42

In [22]:
import random
random.seed(GLOBAL_SEED)

In [23]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Qwen3Initializer, LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [24]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Qwen3Initializer(),
)
llm_loader

In [25]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/config.json


`torch_dtype` is deprecated! Use `dtype` instead!


Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_at

loading weights file /raid_storage/shared_models/Qwen3-8B-Base/model.safetensors.index.json


Instantiating Qwen3ForCausalLM model under default dtype torch.bfloat16.


...

loading file tokenizer_config.json


loading file chat_template.jinja


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/generation_config.json


Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}



In [26]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    sample_lines = sample.splitlines()
    # each sample should be two lines only, the first line is for the inputs and the last is the output
    # we did the join for generalization
    prefix = '\n'.join(sample_lines[:-1])
    prefix = prefix.strip()
    prefix += '\nThe answer is:'
    # outputs are always the last line
    suffix = sample_lines[-1].strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(4500,
 500,
 [('Based on the premise: منحه فرصة للترشح للرئاسة.. قاض يلغي إجراءات ملاحقة دا سيلفا في قضيتين, does the following hypothesis logically follow?\n\nHypothesis: ألغى قاض في المحكمة العليا البرازيلية إجراءات  ضد الرئيس السابق \n\nAnswer with only "entails" or "not entails". No need for extra explanation\nThe answer is:',
   ' entails'),
  ('Based on the premise: زحف نحو الحدود ومظاهرات لا تتوقف.. الأردن يغضب للقدس وفلسطين ودعوات لقطع العلاقات مع إسرائيل, does the following hypothesis logically follow?\n\nHypothesis: احتجاجات لبنانية قرب الحدود تضامنا مع فلسطين\n\nAnswer with only "entails" or "not entails". No need for extra explanation\nThe answer is:',
   ' not entail'),
  ('Welcome to the "Logic Match Game!"\n\nTask: In this game, your challenge is to decide if the statement, "الريال اليمني يتراجع إلى أدنى مستوى في تاريخه أمام الدولار," logically follows from the premise, "العملة اليمنية عند أدنى مستوى أمام الدولار." Think carefully: does the premise support the hypothesi

In [27]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=16,
    eval_batch_size=16,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}'
)

PyTorch: setting up devices


The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/llm_trainer.py:83: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.


Using auto half precision backend



***** Running Evaluation *****


...

  Num examples = 500


  Batch size = 16


{'eval_loss': 0.1724487841129303, 'eval_model_preparation_time': 0.0002, 'eval_runtime': 8.9063, 'eval_samples_per_second': 56.14, 'eval_steps_per_second': 3.593, 'epoch': 5.319148936170213}




Training completed. Do not forget to share your model on huggingface.co/models =)




0.07123827934265137

In [28]:
exit()